In [0]:
df_taxa_bruta_nat_gold = spark.read.table("workspace.validation.taxa_bruta_natalidade")
df_esp_vida_nasc_gold = spark.read.table("workspace.validation.esperanca_vida_nascer")
df_teste_validacao = spark.read.table("workspace.validation.teste_de_validacao")

df_taxa_bruta_nat_gold = df_taxa_bruta_nat_gold.union(df_teste_validacao)

### Filtrando os dados válidos e rejeitados de cada tabela

In [0]:
from pyspark.sql import functions as F

df_taxa_bruta_validos = df_taxa_bruta_nat_gold.filter(F.size("motivos_rejeicao") == 0)
df_taxa_bruta_rejeitados = df_taxa_bruta_nat_gold.filter(F.size("motivos_rejeicao") > 0)

df_esp_vida_validos = df_esp_vida_nasc_gold.filter(F.size("motivos_rejeicao") == 0)
df_esp_vida_rejeitados = df_esp_vida_nasc_gold.filter(F.size("motivos_rejeicao") > 0)


print(f"Taxa Bruta: {df_taxa_bruta_validos.count():,} válidos e {df_taxa_bruta_rejeitados.count():,} rejeitados")
print(f"Esperança Vida: {df_esp_vida_validos.count():,} válidos e {df_esp_vida_rejeitados.count():,} rejeitados")

### Criando tabela GOLD

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

### Tabela de quarentena para saber quais dados foram rejeitados e os motivos

In [0]:
df_quarentena = df_taxa_bruta_rejeitados.union(df_esp_vida_rejeitados)

df_quarentena.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "True") \
.saveAsTable("workspace.gold.quarentena")

print("Tabela (quarentena) criada corretamente.")

### Agregações:

- Join entre taxa de natalidade e esperança de vida por UF e ano, para sabermos os dois indicadores por estado e período
- Média dos indicadores agrupada por região e ano, para sabermos como cada região do Brasil se comporta ao longo do tempo
- Evolução dos indicadores nacionais por ano, para sabermos a tendência geral do Brasil
- Ranking de UFs por esperança de vida, para sabermos quais estados tem maior e menor expectativa de vida
- Ranking de UFs por taxa de natalidade, para sabermos quais estados tem maior e menor taxa de nascimentos
- Comparativo entre UF e Média Nacional, para sabermos quais estados estão acima ou abaixo da média do Brasil em cada indicador

In [0]:
# Agregação 1
nat = df_taxa_bruta_validos.alias("nat")
esp = df_esp_vida_validos.alias("esp")

df_indicadores_uf_ano = nat.join(esp, on=["sg_uf", "co_ano"], how="inner") \
    .select(
        F.col("sg_uf"),
        F.col("co_ano"),
        F.col("nat.no_regiao_brasil"),
        F.col("nat.vl_indicador_calculado_uf").alias("taxa_natalidade_indicador_uf"),
        F.col("nat.vl_indicador_calculado_reg").alias("taxa_natalidade_indicador_reg"),
        F.col("nat.vl_indicador_calculado_br").alias("taxa_natalidade_indicador_br"),
        F.col("esp.vl_indicador_calculado_uf").alias("esperanca_vida_indicador_uf"),
        F.col("esp.vl_indicador_calculado_reg").alias("esperanca_vida_indicador_reg"),
        F.col("esp.vl_indicador_calculado_br").alias("esperanca_vida_indicador_br")
    )

display(df_indicadores_uf_ano.limit(10))

In [0]:
df_indicadores_uf_ano.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "True") \
.saveAsTable("workspace.gold.indicadores_uf_ano")

print("Tabela (indicadores_uf_ano) criada corretamente.")

In [0]:
# Agregação 2
df_media_indicadores_ano = df_indicadores_uf_ano.groupBy("no_regiao_brasil", "co_ano").agg(
    F.avg("taxa_natalidade_indicador_uf").cast("decimal(10,2)").alias("taxa_natalidade_média_indicador_uf_ano"),
    F.avg("taxa_natalidade_indicador_reg").cast("decimal(10,2)").alias("taxa_natalidade_média_indicador_reg_ano"),
    F.avg("taxa_natalidade_indicador_br").cast("decimal(10,2)").alias("taxa_natalidade_média_indicador_br_ano"),
    F.avg("esperanca_vida_indicador_uf").cast("decimal(10,2)").alias("esperanca_vida_média_indicador_uf_ano"),
    F.avg("esperanca_vida_indicador_reg").cast("decimal(10,2)").alias("esperanca_vida_média_indicador_reg_ano"),
    F.avg("esperanca_vida_indicador_br").cast("decimal(10,2)").alias("esperanca_vida_média_indicador_br_ano")
)

display(df_media_indicadores_ano.limit(10))

In [0]:
df_media_indicadores_ano.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "True") \
.saveAsTable("workspace.gold.media_indicadores_ano")

print("Tabela (media_indicadores_ano) criada corretamente.")

In [0]:
# Agregação 3
df_indicadores_br_ano = df_indicadores_uf_ano.groupBy("co_ano").agg(
    F.avg("taxa_natalidade_indicador_br").cast("decimal(10,2)").alias("taxa_natalidade_média_indicador_br_ano"),
    F.avg("esperanca_vida_indicador_br").cast("decimal(10,2)").alias("esperanca_vida_média_indicador_br_ano")
)

display(df_indicadores_br_ano.limit(10))

In [0]:
df_indicadores_br_ano.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "True") \
.saveAsTable("workspace.gold.indicadores_br_ano")

print("Tabela (indicadores_br_ano) criada corretamente.")

In [0]:
# Agregação 4
df_ranking_esp_vida = df_indicadores_uf_ano.groupBy("sg_uf", "no_regiao_brasil").agg(
    F.avg("esperanca_vida_indicador_uf").cast("decimal(10,2)").alias("esperanca_vida_media")).orderBy(F.desc("esperanca_vida_media"))

display(df_ranking_esp_vida)

In [0]:
df_ranking_esp_vida.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "True") \
.saveAsTable("workspace.gold.ranking_esp_vida")

print("Tabela (ranking_esp_vida) criada corretamente.")

In [0]:
# Agregação 5
df_ranking_natalidade = df_indicadores_uf_ano.groupBy("sg_uf", "no_regiao_brasil").agg(
    F.avg("taxa_natalidade_indicador_uf").cast("decimal(10,2)").alias("taxa_natalidade_media")).orderBy(F.desc("taxa_natalidade_media"))

display(df_ranking_natalidade)

In [0]:
df_ranking_natalidade.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "True") \
.saveAsTable("workspace.gold.ranking_natalidade")

print("Tabela (ranking_natalidade) criada corretamente.")

In [0]:
# Agregação 6
df_comparativo_uf_vs_br = df_indicadores_uf_ano.select(
    "sg_uf",
    "co_ano",
    "no_regiao_brasil",
    "taxa_natalidade_indicador_uf",
    "taxa_natalidade_indicador_br",
    ((F.col("taxa_natalidade_indicador_uf") - F.col("taxa_natalidade_indicador_br")) / F.col("taxa_natalidade_indicador_br") * 100).cast("decimal(10,2)").alias("taxa_natalidade_desvio_perc_br"),
    "esperanca_vida_indicador_uf",
    "esperanca_vida_indicador_br",
    ((F.col("esperanca_vida_indicador_uf") - F.col("esperanca_vida_indicador_br")) / F.col("esperanca_vida_indicador_br") * 100).cast("decimal(10,2)").alias("esperanca_vida_desvio_perc_br")
)

display(df_comparativo_uf_vs_br.limit(10))

In [0]:
df_comparativo_uf_vs_br.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "True") \
.saveAsTable("workspace.gold.comparativo_uf_vs_br")

print("Tabela (comparativo_uf_vs_br) criada corretamente.")